<a href="https://colab.research.google.com/github/leorfoletto/car_prediction/blob/main/analise_bolsa_br.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# CÉLULA 1: INSTALAÇÕES E IMPORTS
# =============================================================================
# !pip install yfinance pandas plotly lxml html5lib beautifulsoup4 tqdm --quiet --upgrade

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import pickle, os, time, random
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# --------- CONFIGURAÇÕES GLOBAIS ---------
TAXA_DESCONTO_BAZIN = 0.06
GRAHAM_MULTIPLIER   = 22.5
PEG_THRESHOLD       = 1.0
DY_MINIMO_BARSI     = 0.06

# Limites anti-rate-limit do Yahoo
MAX_WORKERS      = 6          # Nº de threads paralelas (conservador)
REQUEST_DELAY    = 0.15       # Delay aleatório entre requisições
MAX_RETRIES      = 3          # Tentativas por ticker em caso de falha
CACHE_DIR        = '/content/cache_yfinance'
CACHE_VALIDADE_H = 12         # Cache válido por 12h

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"✅ Setup completo. Cache em: {CACHE_DIR}")

In [ ]:
# =============================================================================
# CÉLULA 2: UNIVERSOS DE TICKERS — LISTAS PRÉ-CONSTRUÍDAS E SCRAPERS
# =============================================================================

# --- IBOVESPA e IBrX-100 (B3) ---
# Lista curada das ações mais líquidas da B3 (atualize trimestralmente)
TICKERS_IBOV = [
    'ALOS3.SA','ALPA4.SA','ABEV3.SA','ARZZ3.SA','ASAI3.SA','AURE3.SA','AZUL4.SA',
    'AZZA3.SA','B3SA3.SA','BBSE3.SA','BBDC3.SA','BBDC4.SA','BRAP4.SA','BBAS3.SA',
    'BRKM5.SA','BRFS3.SA','BPAC11.SA','CXSE3.SA','CRFB3.SA','CCRO3.SA','CMIG4.SA',
    'COGN3.SA','CPLE6.SA','CSAN3.SA','CPFE3.SA','CMIN3.SA','CVCB3.SA','CYRE3.SA',
    'ELET3.SA','ELET6.SA','EMBR3.SA','ENGI11.SA','ENEV3.SA','EGIE3.SA','EQTL3.SA',
    'EZTC3.SA','FLRY3.SA','GGBR4.SA','GOAU4.SA','NTCO3.SA','HAPV3.SA','HYPE3.SA',
    'IGTI11.SA','IRBR3.SA','ITSA4.SA','ITUB4.SA','JBSS3.SA','KLBN11.SA','RENT3.SA',
    'LREN3.SA','LWSA3.SA','MGLU3.SA','MRFG3.SA','BEEF3.SA','MRVE3.SA','MULT3.SA',
    'PCAR3.SA','PETR3.SA','PETR4.SA','RECV3.SA','PRIO3.SA','PETZ3.SA','RADL3.SA',
    'RAIZ4.SA','RDOR3.SA','RAIL3.SA','SBSP3.SA','SANB11.SA','STBP3.SA','SMTO3.SA',
    'CSNA3.SA','SLCE3.SA','SUZB3.SA','TAEE11.SA','VIVT3.SA','TIMS3.SA','TOTS3.SA',
    'UGPA3.SA','USIM5.SA','VALE3.SA','VAMO3.SA','VBBR3.SA','VIVA3.SA','WEGE3.SA',
    'YDUQ3.SA'
]

# --- Small Caps B3 adicionais (para chegar a ~150 BR) ---
TICKERS_SMALL_BR = [
    'TRPL4.SA','GRND3.SA','SAPR11.SA','CSMG3.SA','ODPV3.SA','LEVE3.SA','POMO4.SA',
    'UNIP6.SA','KEPL3.SA','WIZC3.SA','DIRR3.SA','TUPY3.SA','SIMH3.SA','PSSA3.SA',
    'GOLL4.SA','CEAB3.SA','AMER3.SA','ARML3.SA','PORT3.SA','BMOB3.SA','BLAU3.SA',
    'PGMN3.SA','CASH3.SA','INTB3.SA','MOVI3.SA','TEND3.SA','RANI3.SA','ROMI3.SA',
    'SEQL3.SA','LOGG3.SA','JALL3.SA','AGRO3.SA','MDIA3.SA','ALUP11.SA','CSED3.SA',
    'MILS3.SA','ENJU3.SA','VIVR3.SA','VLID3.SA','PDGR3.SA','EVEN3.SA','TGMA3.SA',
    'SAPR4.SA','SAPR3.SA','ALUP4.SA','CBAV3.SA','CAML3.SA','QUAL3.SA','GGPS3.SA',
    'ONCO3.SA','BRSR6.SA','GUAR3.SA','ECOR3.SA','BRST3.SA'
]

def obter_sp500():
    """Scrape dinâmico do S&P 500 via Wikipedia."""
    try:
        url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        tabelas = pd.read_html(url)
        sp500 = tabelas[0]['Symbol'].tolist()
        # Corrige tickers com pontos (ex: BRK.B → BRK-B no Yahoo)
        sp500 = [t.replace('.', '-') for t in sp500]
        return sp500
    except Exception as e:
        print(f"⚠️ Falha ao buscar S&P 500: {e}")
        return []


def obter_nasdaq100():
    """Scrape dinâmico do NASDAQ-100."""
    try:
        url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
        tabelas = pd.read_html(url)
        # A tabela relevante pode variar — procura a que tem 'Ticker' ou 'Symbol'
        for t in tabelas:
            cols = [c.lower() for c in t.columns.astype(str)]
            if any('ticker' in c or 'symbol' in c for c in cols):
                col = [c for c in t.columns if 'ticker' in str(c).lower() or 'symbol' in str(c).lower()][0]
                return t[col].tolist()
        return []
    except Exception as e:
        print(f"⚠️ Falha ao buscar NASDAQ-100: {e}")
        return []


def construir_universo(incluir_br=True, incluir_sp500=True,
                        incluir_nasdaq=False, incluir_small_br=True,
                        custom_tickers=None, limite=None):
    """
    Monta a lista final de tickers de acordo com os mercados desejados.

    Parâmetros:
      incluir_br       : IBOV (≈85 ações)
      incluir_small_br : Small caps BR adicionais (≈55 ações)
      incluir_sp500    : S&P 500 (≈500 ações)
      incluir_nasdaq   : NASDAQ-100 (sobreposição grande com SP500)
      custom_tickers   : lista adicional do usuário
      limite           : trunca em N tickers (usar para testes)
    """
    universo = []
    if incluir_br:       universo += TICKERS_IBOV
    if incluir_small_br: universo += TICKERS_SMALL_BR
    if incluir_sp500:    universo += obter_sp500()
    if incluir_nasdaq:   universo += obter_nasdaq100()
    if custom_tickers:   universo += custom_tickers

    # Deduplica preservando ordem
    universo = list(dict.fromkeys(universo))

    if limite:
        universo = universo[:limite]

    print(f"📊 Universo construído: {len(universo)} tickers únicos")
    return universo


# Exemplos de uso:
# universo = construir_universo(incluir_br=True, incluir_sp500=False)   # ~140 BR
# universo = construir_universo(incluir_br=True, incluir_sp500=True)    # ~640 total
# universo = construir_universo(limite=100)                              # teste rápido

In [ ]:
# =============================================================================
# CÉLULA 3: EXTRAÇÃO COM CACHE, RETRY E BACKOFF
# =============================================================================

def _cache_path(ticker: str) -> str:
    return os.path.join(CACHE_DIR, f"{ticker.replace('.', '_')}.pkl")


def _cache_valido(path: str) -> bool:
    """Verifica se o arquivo de cache ainda está dentro da validade."""
    if not os.path.exists(path):
        return False
    mtime = datetime.fromtimestamp(os.path.getmtime(path))
    return datetime.now() - mtime < timedelta(hours=CACHE_VALIDADE_H)


def safe_get(info: dict, key: str, default=np.nan):
    val = info.get(key, default) if info else default
    return np.nan if val is None else val


def _extrair_do_yahoo(ticker: str) -> dict:
    """Núcleo de extração — faz UMA tentativa, sem retry."""
    tk = yf.Ticker(ticker)
    info = tk.info or {}

    # Sanity check: se info vier vazio demais, yahoo falhou silenciosamente
    if len(info) < 5:
        raise ValueError(f"Info insuficiente para {ticker}")

    preco = safe_get(info, 'currentPrice') or safe_get(info, 'regularMarketPrice')
    dy    = safe_get(info, 'dividendYield')
    if pd.notna(dy) and dy > 1:
        dy = dy / 100

    # EBIT via financials (com fallback p/ EBITDA)
    ebit = np.nan
    try:
        fin = tk.financials
        if fin is not None and not fin.empty:
            if 'EBIT' in fin.index:
                ebit = fin.loc['EBIT'].iloc[0]
            elif 'Operating Income' in fin.index:
                ebit = fin.loc['Operating Income'].iloc[0]
    except Exception:
        pass
    if pd.isna(ebit):
        ebit = safe_get(info, 'ebitda')

    # ROIC via balance sheet
    roic = np.nan
    try:
        bs = tk.balance_sheet
        if bs is not None and not bs.empty and pd.notna(ebit):
            equity = bs.loc['Stockholders Equity'].iloc[0] if 'Stockholders Equity' in bs.index else np.nan
            debt   = bs.loc['Total Debt'].iloc[0] if 'Total Debt' in bs.index else np.nan
            if pd.notna(equity) and pd.notna(debt) and (equity + debt) > 0:
                roic = ebit / (equity + debt)
    except Exception:
        pass
    if pd.isna(roic):
        roic = safe_get(info, 'returnOnAssets')

    return {
        'Ticker': ticker,
        'Nome': safe_get(info, 'longName', ticker),
        'Setor': safe_get(info, 'sector', 'N/A'),
        'País': safe_get(info, 'country', 'N/A'),
        'Preço': preco,
        'Market Cap': safe_get(info, 'marketCap'),
        'Enterprise Value': safe_get(info, 'enterpriseValue'),
        'Volume Médio': safe_get(info, 'averageVolume'),
        'P/L': safe_get(info, 'trailingPE'),
        'P/VP': safe_get(info, 'priceToBook'),
        'EV/EBITDA': safe_get(info, 'enterpriseToEbitda'),
        'ROE': safe_get(info, 'returnOnEquity'),
        'ROIC': roic,
        'LPA': safe_get(info, 'trailingEps'),
        'VPA': safe_get(info, 'bookValue'),
        'DY': dy,
        'Div 12M': safe_get(info, 'trailingAnnualDividendRate'),
        'Crescimento Lucros': safe_get(info, 'earningsGrowth'),
        'PEG (yf)': safe_get(info, 'trailingPegRatio') or safe_get(info, 'pegRatio'),
        'EBIT': ebit,
    }


def extrair_ticker_resiliente(ticker: str, usar_cache: bool = True) -> dict:
    """
    Extrai com:
      1. Cache em disco (evita rechamar Yahoo se dados frescos)
      2. Retry com backoff exponencial
      3. Delay aleatório (evita padrão de bot)
    """
    cache_file = _cache_path(ticker)

    # 1) Tenta cache
    if usar_cache and _cache_valido(cache_file):
        try:
            with open(cache_file, 'rb') as f:
                return pickle.load(f)
        except Exception:
            pass  # cache corrompido, segue para yahoo

    # 2) Retry com backoff
    ultima_exc = None
    for tentativa in range(MAX_RETRIES):
        try:
            time.sleep(REQUEST_DELAY + random.random() * 0.1)
            dados = _extrair_do_yahoo(ticker)

            # Salva cache
            try:
                with open(cache_file, 'wb') as f:
                    pickle.dump(dados, f)
            except Exception:
                pass

            return dados
        except Exception as e:
            ultima_exc = e
            # Backoff exponencial: 1s, 2s, 4s
            time.sleep((2 ** tentativa) + random.random())

    # Falhou todas as tentativas — retorna placeholder
    return {'Ticker': ticker, 'erro': str(ultima_exc)}


def extrair_universo(tickers: list, usar_cache: bool = True) -> pd.DataFrame:
    """Extração paralela com barra de progresso."""
    resultados = []
    falhas = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extrair_ticker_resiliente, t, usar_cache): t
                   for t in tickers}

        for fut in tqdm(as_completed(futures), total=len(tickers),
                         desc='📡 Extraindo dados'):
            res = fut.result()
            if 'erro' in res:
                falhas.append(res['Ticker'])
            else:
                resultados.append(res)

    df = pd.DataFrame(resultados)
    if 'Ticker' in df.columns:
        df = df.set_index('Ticker')

    print(f"\n✅ Sucesso: {len(resultados)} | ❌ Falhas: {len(falhas)}")
    if falhas and len(falhas) <= 20:
        print(f"   Tickers com falha: {falhas}")
    elif falhas:
        print(f"   {len(falhas)} falhas (muitas para listar)")

    return df


def limpar_cache():
    """Utilitário para forçar re-download na próxima execução."""
    import shutil
    if os.path.exists(CACHE_DIR):
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR)
    print("🗑️  Cache limpo.")

In [ ]:
# =============================================================================
# CÉLULA 4: FILTROS DE QUALIDADE — EVITAR LIXO NO RANKING
# =============================================================================

def filtrar_qualidade(df: pd.DataFrame,
                      market_cap_min: float = 1e9,
                      volume_min: float = 1e5,
                      exigir_fundamentos: bool = True,
                      min_campos_validos: int = 8) -> pd.DataFrame:
    """
    Remove ações com dados insuficientes ou pouca liquidez.

    Parâmetros:
      market_cap_min     : em USD/BRL (1e9 = 1 bilhão)
      volume_min         : volume médio mínimo
      exigir_fundamentos : exige LPA, VPA, ROE não-nulos
      min_campos_validos : mínimo de campos não-NaN nas colunas críticas
    """
    n_antes = len(df)
    df = df.copy()

    # Filtro 1: liquidez
    if market_cap_min:
        df = df[df['Market Cap'].fillna(0) >= market_cap_min]
    if volume_min:
        df = df[df['Volume Médio'].fillna(0) >= volume_min]

    # Filtro 2: fundamentos mínimos
    if exigir_fundamentos:
        df = df.dropna(subset=['LPA', 'VPA'], how='any')

    # Filtro 3: preenchimento geral
    cols_criticas = ['Preço','P/L','P/VP','ROE','LPA','VPA','DY',
                     'Market Cap','Enterprise Value']
    df['_campos_validos'] = df[cols_criticas].notna().sum(axis=1)
    df = df[df['_campos_validos'] >= min_campos_validos]
    df = df.drop(columns='_campos_validos')

    n_depois = len(df)
    print(f"🧹 Filtro de qualidade: {n_antes} → {n_depois} "
          f"(removidas {n_antes - n_depois} ações com dados insuficientes)")
    return df


def diagnostico_cobertura(df: pd.DataFrame):
    """Mostra % de preenchimento de cada campo — diagnóstico."""
    cobertura = (df.notna().sum() / len(df) * 100).sort_values(ascending=False)
    cob_df = pd.DataFrame({'% Preenchimento': cobertura}).round(1)
    print("\n📊 Cobertura de dados por campo:")
    return cob_df

In [ ]:
# =============================================================================
# CÉLULA 5: 4 ESTRATÉGIAS + RANKING COMPOSTO
# =============================================================================

def aplicar_graham(df):
    mask = (df['LPA'] > 0) & (df['VPA'] > 0)
    vi = np.where(mask, np.sqrt(GRAHAM_MULTIPLIER * df['VPA'] * df['LPA']), np.nan)
    df['VI Graham']            = vi
    df['Margem Segurança (%)'] = (df['VI Graham'] - df['Preço']) / df['Preço'] * 100
    df['✓ Graham']             = df['Margem Segurança (%)'] > 0
    return df


def aplicar_lynch(df):
    cresc_pct = df['Crescimento Lucros'] * 100
    peg_calc = np.where(
        (df['P/L'] > 0) & (cresc_pct > 0),
        df['P/L'] / cresc_pct,
        np.nan
    )
    df['PEG'] = df['PEG (yf)'].fillna(pd.Series(peg_calc, index=df.index))
    df['✓ Lynch'] = (df['PEG'] > 0) & (df['PEG'] < PEG_THRESHOLD)
    return df


def aplicar_formula_magica(df, top_n_pct=0.3):
    """
    Com 500 ações, 'passa' na Fórmula Mágica quem está no top 30% do
    ranking composto (EY + ROIC). Parametrizável.
    """
    df['Earnings Yield'] = np.where(
        (df['Enterprise Value'] > 0) & df['EBIT'].notna(),
        df['EBIT'] / df['Enterprise Value'],
        np.nan
    )
    # Ranking apenas entre ações que têm ambas as métricas
    mask_valido = df['Earnings Yield'].notna() & df['ROIC'].notna()

    df['Rank EY']   = df.loc[mask_valido, 'Earnings Yield'].rank(ascending=False, method='min')
    df['Rank ROIC'] = df.loc[mask_valido, 'ROIC'].rank(ascending=False, method='min')
    df['Rank Mágica Total'] = df['Rank EY'] + df['Rank ROIC']

    # Top N% vencem
    threshold = df['Rank Mágica Total'].quantile(top_n_pct)
    df['✓ Fórmula Mágica'] = df['Rank Mágica Total'] <= threshold
    return df


def aplicar_bazin_barsi(df):
    df['Preço Teto'] = df['Div 12M'] / TAXA_DESCONTO_BAZIN
    df['Upside Bazin (%)'] = (df['Preço Teto'] - df['Preço']) / df['Preço'] * 100
    df['✓ Bazin/Barsi'] = (df['Preço'] < df['Preço Teto']) & (df['DY'] > DY_MINIMO_BARSI)
    return df


def score_final(df):
    cols = ['✓ Graham', '✓ Lynch', '✓ Fórmula Mágica', '✓ Bazin/Barsi']
    df['Score (0-4)'] = df[cols].fillna(False).sum(axis=1)

    # Ranking composto percentil-based (mais robusto em larga escala)
    # Normaliza cada métrica em [0,1] e soma com pesos
    metricas_peso = {
        'Margem Segurança (%)':  0.25,
        'ROIC':                  0.25,
        'Earnings Yield':        0.20,
        'DY':                    0.15,
        'ROE':                   0.15,
    }
    score_continuo = pd.Series(0.0, index=df.index)
    for col, peso in metricas_peso.items():
        if col in df.columns:
            pct = df[col].rank(pct=True)   # percentil 0-1
            score_continuo += pct.fillna(0) * peso

    df['Score Contínuo'] = (score_continuo * 100).round(2)
    return df


def rodar_screening(tickers: list, filtros: dict = None) -> pd.DataFrame:
    """Pipeline end-to-end."""
    filtros = filtros or {}

    print(f"\n🔄 Etapa 1/3: Extraindo dados de {len(tickers)} tickers...")
    df = extrair_universo(tickers)

    print(f"\n🔄 Etapa 2/3: Aplicando filtros de qualidade...")
    df = filtrar_qualidade(df, **filtros)

    print(f"\n🔄 Etapa 3/3: Aplicando estratégias...")
    df = aplicar_graham(df)
    df = aplicar_lynch(df)
    df = aplicar_formula_magica(df)
    df = aplicar_bazin_barsi(df)
    df = score_final(df)

    print(f"\n✅ Screening concluído! {len(df)} ações no ranking final.")
    return df

In [ ]:
# =============================================================================
# CÉLULA 6: TABELAS FORMATADAS + FILTROS POR SCORE
# =============================================================================

def destacar_bool(val):
    if pd.isna(val):  return 'background-color: #f0f0f0; color: #888'
    if val is True:   return 'background-color: #90EE90; color: #003300; font-weight: bold'
    if val is False:  return 'background-color: #FFB6B6; color: #660000'
    return ''


def destacar_score(val):
    if pd.isna(val): return ''
    cores = {0:'#FFB6B6', 1:'#FFD699', 2:'#FFF2B3', 3:'#C2F0C2', 4:'#66CC66'}
    return f'background-color: {cores.get(int(val), "#FFFFFF")}; font-weight: bold'


def formatar_tabela(df, top_n: int = 30):
    """Exibe Top N do ranking composto com styling."""
    cols_bool = ['✓ Graham','✓ Lynch','✓ Fórmula Mágica','✓ Bazin/Barsi']
    cols_show = ['Nome','Setor','Preço','P/L','P/VP','ROE','ROIC','DY',
                 'Margem Segurança (%)','✓ Graham','PEG','✓ Lynch',
                 'Earnings Yield','✓ Fórmula Mágica','Upside Bazin (%)',
                 '✓ Bazin/Barsi','Score (0-4)','Score Contínuo']
    cols_show = [c for c in cols_show if c in df.columns]

    df_top = (df.sort_values(['Score (0-4)','Score Contínuo'], ascending=False)
                .head(top_n)[cols_show])

    styled = (df_top.style
              .applymap(destacar_bool, subset=[c for c in cols_bool if c in df_top.columns])
              .applymap(destacar_score, subset=['Score (0-4)'])
              .background_gradient(subset=['Score Contínuo'], cmap='RdYlGn')
              .format({
                  'Preço':'{:.2f}','P/L':'{:.2f}','P/VP':'{:.2f}',
                  'ROE':'{:.2%}','ROIC':'{:.2%}','DY':'{:.2%}',
                  'Earnings Yield':'{:.2%}',
                  'Margem Segurança (%)':'{:.1f}%',
                  'PEG':'{:.2f}','Upside Bazin (%)':'{:.1f}%',
                  'Score (0-4)':'{:.0f}','Score Contínuo':'{:.1f}',
              }, na_rep='—')
              .set_caption(f'🏆 Top {top_n} do Ranking Multi-Estratégia'))
    return styled


def filtrar_por_estrategia(df, estrategia: str):
    """Mostra só as que passaram numa estratégia específica."""
    col = f'✓ {estrategia}'
    return df[df[col] == True].sort_values('Score Contínuo', ascending=False)


def resumo_por_setor(df):
    """Agrupa o score médio por setor — ótimo para encontrar setores baratos."""
    resumo = (df.groupby('Setor')
                .agg(N_Empresas=('Preço','count'),
                     Score_Medio=('Score (0-4)','mean'),
                     ROIC_Medio=('ROIC','median'),
                     PL_Mediano=('P/L','median'),
                     DY_Medio=('DY','median'))
                .sort_values('Score_Medio', ascending=False))
    return resumo.style.background_gradient(cmap='RdYlGn', subset=['Score_Medio'])

In [ ]:
# =============================================================================
# CÉLULA 7: GRÁFICOS INTERATIVOS (Plotly) PARA UNIVERSO AMPLO
# =============================================================================

def plot_magica_vs_dividendos(df, destacar_top=20):
    """Scatter ROIC × EY com DY como tamanho. Adaptado para centenas de pontos."""
    df_plot = df.reset_index().dropna(subset=['ROIC','Earnings Yield']).copy()
    df_plot['DY_size'] = df_plot['DY'].fillna(0).clip(lower=0.001) * 100

    # Clipa outliers para visualização (sem alterar dados)
    df_plot['ROIC_plot'] = df_plot['ROIC'].clip(lower=-0.3, upper=0.6)
    df_plot['EY_plot']   = df_plot['Earnings Yield'].clip(lower=-0.1, upper=0.3)

    fig = px.scatter(
        df_plot,
        x='ROIC_plot', y='EY_plot',
        size='DY_size',
        color='Score (0-4)',
        color_continuous_scale='RdYlGn',
        hover_name='Ticker',
        hover_data={'Nome':True,'Setor':True,'Preço':':.2f',
                    'ROIC':':.2%','Earnings Yield':':.2%','DY':':.2%',
                    'Score (0-4)':True,'ROIC_plot':False,'EY_plot':False,'DY_size':False},
        size_max=40,
        title=f'<b>Fórmula Mágica × Dividendos — {len(df_plot)} ações</b><br>'
              f'<sup>Top-direita = zona Greenblatt. Bolhas grandes = alto DY.</sup>',
    )

    # Destaca o top N com anotações
    top = df_plot.nlargest(destacar_top, 'Score Contínuo')
    for _, row in top.iterrows():
        fig.add_annotation(x=row['ROIC_plot'], y=row['EY_plot'],
                           text=row['Ticker'], showarrow=False,
                           font=dict(size=9, color='black'), yshift=10)

    fig.add_hline(y=df_plot['EY_plot'].median(),  line_dash='dash', line_color='gray', opacity=0.4)
    fig.add_vline(x=df_plot['ROIC_plot'].median(), line_dash='dash', line_color='gray', opacity=0.4)

    fig.update_layout(
        xaxis=dict(tickformat='.0%', title='ROIC'),
        yaxis=dict(tickformat='.0%', title='Earnings Yield'),
        template='plotly_white', height=700,
    )
    fig.show()


def plot_heatmap_setorial(df):
    """Heatmap: quantas empresas de cada setor passaram em cada estratégia."""
    cols = ['✓ Graham','✓ Lynch','✓ Fórmula Mágica','✓ Bazin/Barsi']
    heat = (df.groupby('Setor')[cols].sum().astype(int))
    heat.columns = [c.replace('✓ ','') for c in heat.columns]

    fig = px.imshow(
        heat, text_auto=True, aspect='auto',
        color_continuous_scale='Greens',
        title='<b>Aprovações por Setor × Estratégia</b>',
        labels=dict(x='Estratégia', y='Setor', color='Nº aprovações'),
    )
    fig.update_layout(height=max(400, len(heat)*25), template='plotly_white')
    fig.show()


def plot_distribuicao_scores(df):
    """Histograma do Score Contínuo."""
    fig = px.histogram(
        df.reset_index(), x='Score Contínuo', color='Score (0-4)',
        nbins=30, color_discrete_sequence=px.colors.sequential.RdYlGn,
        title='<b>Distribuição do Score Contínuo no Universo</b>',
    )
    fig.update_layout(template='plotly_white', height=450, bargap=0.05)
    fig.show()


def plot_top_por_estrategia(df, top_n=15):
    """Barras horizontais: top N em cada uma das 4 estratégias."""
    from plotly.subplots import make_subplots

    specs = [
        ('Graham',         'Margem Segurança (%)', '%'),
        ('Lynch',          'PEG',                  'PEG (↓ melhor)'),
        ('Fórmula Mágica', 'Rank Mágica Total',    'Rank (↓ melhor)'),
        ('Bazin/Barsi',    'Upside Bazin (%)',     '%'),
    ]

    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=[f'Top {top_n} — {s[0]}' for s in specs])

    for i, (nome, col, _) in enumerate(specs):
        row, col_idx = (i // 2) + 1, (i % 2) + 1
        aprovados = df[df[f'✓ {nome}'] == True].copy()
        if aprovados.empty: continue

        asc = nome in ('Lynch','Fórmula Mágica')  # menor é melhor
        top = aprovados.nsmallest(top_n, col) if asc else aprovados.nlargest(top_n, col)

        fig.add_trace(
            go.Bar(x=top[col], y=top.index, orientation='h',
                   marker_color='#2E8B57', showlegend=False,
                   text=top[col].round(2), textposition='outside'),
            row=row, col=col_idx
        )

    fig.update_layout(height=800, template='plotly_white',
                      title_text='<b>Top 15 por Estratégia</b>')
    fig.show()

In [ ]:
# =============================================================================
# CÉLULA 8: EXECUÇÃO DO PIPELINE
# =============================================================================

# ---- ESCOLHA SEU UNIVERSO ----
universo = construir_universo(
    incluir_br       = True,   # IBOV ≈ 85
    incluir_small_br = True,   # +55 small caps BR
    incluir_sp500    = True,   # +500 americanas
    incluir_nasdaq   = False,
    # limite=100,              # descomente para teste rápido
)

# ---- RODA O SCREENING ----
filtros = dict(
    market_cap_min     = 1e9,      # 1B (ajuste para 5e8 se quiser mid/small caps)
    volume_min         = 1e5,
    exigir_fundamentos = True,
    min_campos_validos = 8,
)

df_final = rodar_screening(universo, filtros=filtros)

# ---- DIAGNÓSTICO ----
print("\n")
diagnostico_cobertura(df_final)

In [ ]:
# Célula 8b — Visualizações
formatar_tabela(df_final, top_n=30)

In [ ]:
# Célula 8c
print("\n🔍 Top 20 por Fórmula Mágica (Greenblatt):")
filtrar_por_estrategia(df_final, 'Fórmula Mágica').head(20)[
    ['Nome','Setor','ROIC','Earnings Yield','P/L','DY','Score (0-4)']
]

In [ ]:
# Célula 8d
resumo_por_setor(df_final)